In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(p for p in candidates if (p / "pyproject.toml").exists() and (p / "src").exists())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})
PROJECT_ROOT

from scipy.signal import resample
from mpl_toolkits.mplot3d import axes3d

In [ ]:
%reload_ext autoreload
%autoreload 2
from ica_utils import *

In [ ]:
traces = np.load('fluo_220127_F4_run2.npy')
tail_angle= np.load('220127_F4_run2_tail_angle.npy')
all_background = (np.load('all_background.npy', allow_pickle=True))
all_positions = np.load('all_positions.npy', allow_pickle=True)

In [ ]:
# resampling behavior to same number of samples as the traces

aa = resample(np.abs(tail_angle),3176,domain='time')
plt.figure(figsize=(20,2))
plt.plot(aa)
aa = aa[np.newaxis,:]

In [ ]:
f_s = 5.3

In [ ]:
traces.shape

In [ ]:
n = eig_dec(traces)

In [ ]:
ic_comps,IC_ft,A,mean = ica_dec(traces,n,t=0.0001,max_=500)

In [ ]:
IC_ft.shape

In [ ]:
def plott_ics(ics):        
    fig,ax = plt.subplots(ics.shape[1],1,figsize=(15,1.3*ics.shape[1]))
    for i in range(ics.shape[1]):
        ax[i].vlines(x=[562,675,1103,1963,2162,2295,2296,2297,472,473,478,447,697,1020,2581,2600],ymin=ics.T[i,:].min(),ymax=ics.T[i,:].max(),ls='--',color='g',lw = .7)
        ax[i].plot((ics.T[i,:]),label='{}'.format(i),lw=.6)
        ax[i].legend()
        
def plottings_spectrals3(n_clus,alll):
    fig,ax=plt.subplots(1,n_clus,figsize=(15,3))
    for i in range(n_clus):
        group = IC_ft[np.where(alll[:,3] == i)]
        for j in range(group.shape[0]):
            ax[i].plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2,3176)),group[j,:])
        ax[i].plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2,3176)),group.mean(0),color='black',lw=1,label='mean')
        ax[i].set_xlim([0,f_s/2])
        ax[i].set_ylim([0,5])
        ax[i].set_title('cl {} with {}'.format(i,group.shape[0]))
        ax[i].legend()

def plottings_logSpectral3(n_clus,alll):
    plt.figure(figsize=(15,8))
    for i in range(n_clus):
        group = IC_ft[np.where(alll[:,3] == i)]
        plt.plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2,3176)),np.log(group.mean(0)),lw=1,label='{}'.format(i))
        plt.xlim([0,.5])
        plt.ylim([0,2.5])
        plt.legend()

In [ ]:
plott_ics(ic_comps)

In [ ]:
plot_FT_spectrals(ic_comps,f_s,n)

In [ ]:
np.save('new_ics.npy',ic_comps)

In [ ]:
plt.plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2, 3176)),IC_ft.T)
plt.xlim(0,3)

#### Seven clusters

In [ ]:
new_mat7, predictions7= cluster(ic_comps,7,f_s)

In [ ]:
%matplotlib notebook
plot_clusters(new_mat7,predictions7)

In [ ]:
al7 = np.c_[new_mat7.round(2),predictions7.round(1),np.arange(n)]
al7

In [ ]:
%matplotlib notebook
plottings_logSpectral3(7,al7)

In [ ]:
plottings_spectrals3(7,al7)

In [ ]:
%matplotlib inline
unique,count = np.unique(predictions7,return_counts=True)
plt.stem(unique,count)
plt.title('{} ICs'.format(np.sum(count)))
plt.xlabel('clusters')
plt.ylabel('count')
plt.grid()

In [ ]:
order = [6,2,5,1,4,3,0]

In [ ]:
### for 6 and 2
%matplotlib inline
ic_ = ic_comps.copy()
new_idx = np.delete(np.arange(n),np.r_[np.where(predictions7==2)[0],np.where(predictions7==6)[0]])
ic_[:,new_idx] = 0
cleaned_62 = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned_62.shape[1],1,figsize=(15,1.2*cleaned_62.shape[1]))
for i in range(cleaned_62.shape[1]):
    ax[i].plot(cleaned_62[:,i]+.75,label=f'cleaned{i}',color='blue',lw=.7)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.7)
    ax[i].vlines(x=[562,675,1103,1963,2162,2295,2296,2297,472,473,478,447,697,1020,2581,2600],ymin=traces[i,:].min(),ymax=traces[i,:].max(),ls='--',color='g')
    ax[i].legend()

In [ ]:
### for 6, 2 and 5
%matplotlib inline
ic_ = ic_comps.copy()
new_idx = np.delete(np.arange(n),np.r_[np.where(predictions7==2)[0],np.where(predictions7==6)[0],np.where(predictions7==5)[0]])
ic_[:,new_idx] = 0
cleaned_625 = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned_625.shape[1],1,figsize=(15,1.2*cleaned_625.shape[1]))
for i in range(cleaned_625.shape[1]):
    ax[i].plot(cleaned_625[:,i]+.75,label=f'cleaned{i}',color='blue',lw=.7)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.7)
    ax[i].vlines(x=[562,675,1103,1963,2162,2295,2296,2297,472,473,478,447,697,1020,2581,2600],ymin=traces[i,:].min(),ymax=traces[i,:].max(),ls='--',color='g')
    ax[i].legend()

In [ ]:
### for 6, 2, 1 and 5
%matplotlib inline
ic_ = ic_comps.copy()
new_idx = np.delete(np.arange(n),np.r_[np.where(predictions7==2)[0],np.where(predictions7==6)[0],np.where(predictions7==5)[0],np.where(predictions7==1)[0]])
ic_[:,new_idx] = 0
cleaned_6251 = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned_6251.shape[1],1,figsize=(15,1.2*cleaned_6251.shape[1]))
for i in range(cleaned_6251.shape[1]):
    ax[i].plot(cleaned_6251[:,i]+.75,label=f'cleaned{i}',color='blue',lw=.7)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.7)
    ax[i].vlines(x=[562,675,1103,1963,2162,2295,2296,2297,472,473,478,447,697,1020,2581,2600],ymin=traces[i,:].min(),ymax=traces[i,:].max(),ls='--',color='g')
    ax[i].legend()

In [ ]:
np.save('cleaned_4.npy',cleaned_6251.T)
np.save('cleaned_3.npy',cleaned_625.T)

In [ ]:
order = [6,2,5,1,4,3,0]

In [ ]:
# %matplotlib inline
# for k in order:
ic_ = ic_comps.copy()
idx = np.where(predictions7!=0)
ic_[:,idx[0]] = 0
cleaned = np.dot(ic_, A.T) + mean
# np.save(f'cleaned_{k}.npy',cleaned.T)
fig,ax = plt.subplots(cleaned.shape[1],1,figsize=(15,1.2*cleaned.shape[1]))
for i in range(cleaned.shape[1]):
    ax[i].plot(cleaned[:,i]+.75,label=f'cleaned{i}',color='blue',lw=.7)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.7)
    ax[i].vlines(x=[939,1360,1361,1434,1437,1689,1690,1705,1991,1992],ymin=traces[i,:].min(),ymax=traces[i,:].max(),lw=.7,ls='--',color='g')
    ax[i].legend()

In [ ]:
idx = np.where(predictions7==0)[0]
len(idx)
A_0 = A[:,idx]
A_0.shape

In [ ]:
def plot_projec(mixing,centers):
    m,n = mixing.min(),mixing.max()
    j = np.abs(np.array([m,n])).max()
    for i in range(mixing.shape[0]):
        plt.figure(figsize=(5.5,4))
        plt.imshow(background,'gray')
        new = np.zeros_like(mixing)
        new[i,:] = mixing[:,i]
        for a in range(centers.shape[0]):
            plt.scatter(centers[:,0],centers[:,1],s=40,c=new[i,:],cmap ='seismic',vmin=-j,vmax=j)
        plt.title('Column {} of A'.format(i))
        plt.colorbar()
        
def plot_projections(corr,centers):
    m,n = corr.min(),corr.max()
    j = np.abs(np.array([m,n])).max()
    for i in range(corr.shape[1]):
        fig = plt.figure(figsize =(10,5.5))
        ax = fig.add_subplot(111,projection="3d")
        p = ax.scatter(centers[:,0],centers[:,1],centers[:,2],marker='o',s=10,c=corr[:,i],cmap ='seismic',vmin=-j,vmax=j)
        fig.colorbar(p)

In [ ]:
plot_projections(A_0,all_positions)

In [ ]:
stop

### Correlations projections

In [ ]:
def plot_projections(corr,centers):
    fig = plt.figure(figsize = (14,8))
    ax = fig.add_subplot(111,projection="3d")
    p = ax.scatter(centers[:,0],centers[:,1],centers[:,2],marker='o',s=30,c=corr,cmap ='seismic',vmin=-.5,vmax=.5)
    fig.colorbar(p)

In [ ]:
jj = np.array([116,158,16,2])

In [ ]:
correlations_06 = np.zeros(traces.shape[0])
for i in range(cleaned_063.shape[1]):
    correlations_06[i]= np.corrcoef(aa,cleaned_06[:,i])[1,0]

In [ ]:
%matplotlib notebook
plot_projections(correlations_06,all_positions)

In [ ]:
fig,ax = plt.subplots(1,len(jj),figsize=(15,3))
for i,j in enumerate(jj):
    ax[i].scatter(aa[0],cleaned_06[:,j],s=10,marker='.')
    ax[i].set_xlabel('behavior')
    ax[i].set_ylabel('cleaned traces')
    ax[i].set_title(f'{correlations_06[j]}')

In [ ]:
correlations_063 = np.zeros(traces.shape[0])
for i in range(cleaned_063.shape[1]):
    correlations_063[i] = np.corrcoef(aa,cleaned_063[:,i])[1,0]

In [ ]:
%matplotlib notebook
plot_projections(correlations_063,all_positions)

In [ ]:
fig,ax = plt.subplots(1,len(jj),figsize=(15,3))
for i,j in enumerate(jj):
    ax[i].scatter(aa[0],cleaned_063[:,j],s=10,marker='.')
    ax[i].set_xlabel('behavior')
    ax[i].set_ylabel('cleaned traces')
    ax[i].set_title(f'{correlations_063[j]}')

### Using manually selected ICs

In [ ]:
aaa = np.array([0,1,2,4,7,9,10,11,12,14,16,17,18,19,21,22,23,24,25,
                26,27,28,29,30,31,33,34,35,37,39,41,42,43,44,47,48,49,
                50,51,52,53,54,56,57,59,60,61,63,64,65,66,68,69,
                70,72,73,74,77,78,79,80,81,82,83,84,85,87,88,89])

In [ ]:
%matplotlib inline
idx_toZero = np.delete(np.arange(n),aaa)
ic_ = ic_comps.copy()
ic_[:,idx_toZero] = 0
cleaned_ = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned_.shape[1],1,figsize=(15,1.2*cleaned_.shape[1]))
for i in range(cleaned_.shape[1]):
    ax[i].plot(cleaned_[:,i]+3,label=f'cleaned{i}',color='blue',lw=.5)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.5)
    ax[i].vlines(x=[562,675,1103,1963,2162,2295,2296,2297,472,473,478,447,697,1020,2581,2600],ymin=traces[i,:].min(),ymax=traces[i,:].max(),ls='--',color='g',lw=.6)
    ax[i].legend()

In [ ]:
correlations_ = np.zeros(traces.shape[0])
for i in range(cleaned_.shape[1]):
    correlations_[i] = np.corrcoef(aa,cleaned_[:,i])[1,0]

In [ ]:
%matplotlib notebook
plot_projections(correlations_,all_positions)

In [ ]:
fig,ax = plt.subplots(1,len(jj),figsize=(15,3))
for i,j in enumerate(jj):
    ax[i].scatter(aa[0],cleaned_[:,j],s=10,marker='.')
    ax[i].set_xlabel('behavior')
    ax[i].set_ylabel('cleaned traces')
    ax[i].set_title(f'{correlations_[j]}')